# 2) 2-Minute Rescue Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

When a student is stuck, select one 2-minute rescue script and track which scripts helped.

## Simple rules (policy)

- Pick one rescue script each time.
- Ask “Did it help? yes/no”.
- Track helped/total stats per script in memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "02_rescue_memory.json"

SCRIPTS = [
    ("Rubber Duck", [
        "Explain the problem out loud in 3 sentences.",
        "Name the exact input and expected output.",
        "Identify the first point you feel uncertain.",
    ]),
    ("Minimal Example", [
        "Create the smallest input that triggers the bug.",
        "Remove everything not needed to reproduce it.",
        "Test again with only the minimal case.",
    ]),
    ("First Error First", [
        "Find the first incorrect value in your trace/output.",
        "Ask: what line produced it?",
        "Check assumptions right before that line.",
    ]),
    ("Base Case Check", [
        "If recursion: write the base case in plain English.",
        "Test the base case with 1 simple input.",
        "Then test 1 step above base case.",
    ]),
    ("Print & Probe", [
        "Add ONE print/log at the key decision point.",
        "Run once and observe the value.",
        "Remove the print after learning.",
    ]),
]

def pick_script(memory: Dict[str, Any]) -> int:
    stats = memory.get("stats", {})
    names = [s[0] for s in SCRIPTS]
    weights = []
    for n in names:
        d = stats.get(n, {"helped": 0, "total": 0})
        helped, total = int(d["helped"]), int(d["total"])
        weights.append((helped + 1) / (total + 2))  # smoothed
    return random.choices(range(len(SCRIPTS)), weights=weights, k=1)[0]

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})
    idx = pick_script(memory)
    name, steps = SCRIPTS[idx]
    memory["last_script"] = name
    msg = "2-Minute Rescue: {}\n{}".format(name, "\n".join([f"- {s}" for s in steps]))
    return Action("notify", {"message": msg})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("2-Minute Rescue Agent is running.")
    Tools.notify("Describe what you're stuck on. Type 'stop' to end.")

    user_text = Tools.ask("What's blocking you?")
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved rescue stats. Bye!")
            break

        ans = Tools.ask("Did it help? (yes/no)")
        stats = memory.get("stats", {})
        name = memory.get("last_script", "unknown")
        d = stats.get(name, {"helped": 0, "total": 0})
        d["total"] = int(d["total"]) + 1
        if ans.lower().startswith("y"):
            d["helped"] = int(d["helped"]) + 1
        stats[name] = d
        memory["stats"] = stats

        env.execute(Action("save_memory", {}), memory)
        user_text = Tools.ask("Another block? (or stop)")

run_agent()


## Easy extensions

- Add a countdown timer (demo mode to shorten).
- Let students add their own rescue scripts.
- Show best success-rate script.